# 🌊 Run Continuum Hydrological Model
<br>
<img style="float: left; padding-right: 15px; padding-left: 0px;" src="../sources/images/logo_continuum.png" width="260px" align=”left” >

<div style="text-align: justify">
This notebook loads parameters from a configuration file (`[domain]_hmc.json`), saved in the `settings` folder, process the data to create the forcing files and tun the Continuum Hydrological Model.

## 🔧 Preliminary Setup

This section handles the initial configuration, including setting the project file, importing necessary libraries, and preparing the working environment.

#### Specify Configuration File

This cell defines the name of the configuration file to be used for the entire workflow. The configuration file, `shebele_hmc.json` in this case, contains all the parameters for the project.

In [1]:
# Set the name of the settings file
settings_file = "awash_hmc2.json"

print(f"Using settings file: {settings_file}")

Using settings file: awash_hmc2.json


#### Import Required Libraries

This cell imports all the Python libraries necessary to run the notebook.

In [2]:
import json
import os
import sys
import rioxarray as rxr
from pathlib import Path
from datetime import datetime

lib_path = os.path.join(str(Path().cwd().parent),"sources","libraries")
sys.path.append(lib_path)

import shybox_tools, hmc_tools

## 🛑 **WARNING! All required data must be in the `hmc_data` folder before proceeding** 🛑
The configurazion file contains the path of the different folder required to run the model.

The `hmc_data` foder must contain the following subfolder:
- /data_geo/point/
- /data_geo/gridded/
- /data_forcing/gridded/
- /data_forcing/point/
- /data_forcing/time_series/

## 📁 Setup folders

Load the configuration and setup the required folders

In [3]:
# Define the paths of the required directories
workspace_path = str(Path().cwd().parent)
data_path = os.path.join(str(Path().cwd().parent.parent),"data")

# Load configuration from the JSON file
with open(os.path.join(workspace_path, "settings", settings_file), 'r') as f:
    config = json.load(f)

print(f"Configuration loaded from settings file: {settings_file}")

# check the folder structure, create the folders if needed
path_settings = config['path']

# check input folder existance
input_path={}
input_path["data_geo_point"] = os.path.join(data_path, path_settings["hmc_data"], "data_geo", "point")
input_path["data_geo_gridded"] = os.path.join(data_path, path_settings["hmc_data"], "data_geo", "gridded")
input_path["data_forcing_point"] = os.path.join(data_path, path_settings["hmc_data"], "data_forcing", "point")
input_path["data_forcing_gridded"] = os.path.join(data_path, path_settings["hmc_data"], "data_forcing", "input")
input_path["data_forcing_time_series"] = os.path.join(data_path, path_settings["hmc_data"], "data_forcing", "time_series")

for key in input_path.keys():
    if not os.path.isdir(input_path[key]):
        if key in ["data_forcing_point" , "data_forcing_time_series"]:
            print("WARNING! Path of " + key + " is not found... But is not mandatory!")
        else:
            raise RuntimeError(f"Input folder {input_path[key]} not found")

print("Folders setup completed");

Configuration loaded from settings file: awash_hmc2.json
WARNING! Path of data_forcing_point is not found... But is not mandatory!
WARNING! Path of data_forcing_time_series is not found... But is not mandatory!
Folders setup completed


## 🗺️ Data Preparation

Prepare the forcing data 

In [ ]:
# Specify run dates
date_start = "2022-07-01 00:00"
date_end = "2022-07-02 00:00"

# Prepare continuum data
shybox_tools.prepare_hmc_data(workspace_path, config, date_start, date_end)
print("✅ Forcing files ready!")

# Clean temporary files
try:
    shutil.rmtree(str(Path().cwd().parent) + "/notebook_continuum/tmp")
except:
    pass

## ▶️ Model Execution

### *Settings*
The following settings can be modified:

**Parameters**
- `"dUc": 25` → Friction coefficient in channel *[10–100 m^0.5 s⁻¹]* – used only if `{domain}.uc.txt` is not in the gridded files  
- `"dUh": 0.6` → Flow motion coefficient in hillslopes *[1–20 s⁻¹]* – used only if `{domain}.uh.txt` is not in the gridded files
- `"dCt": 0.47` → Mean field capacity *[0.1–0.9]* – used only if `{domain}.ct.txt` is not in the gridded files  
- `"dSoil_vmax": 700` → Soil total storage capacity (gravitational + capillary) *[300–1500 mm]* – used only if `{domain}.soil_vmax.txt` is not in the gridded files  
- `"dSoil_ksat_drain": 3` → Soil saturated vertical hydraulic conductivity (drainage) *[0.5–50 mm·h⁻¹]* – used only if `{domain}.soil_ksat_drain.txt` is not in the gridded files  
- `"dSoil_ksat_infilt": 9` → Soil saturated vertical hydraulic conductivity (infiltration) *[0.5–50 mm·h⁻¹]* – used only if `{domain}.soil_ksat_infilt.txt` is not in the gridded files
- `"dKSatRatio": 1` → Anisotropy between vertical and horizontal saturated conductivity *[1–3]*
- `"dWTableHbr": 600` → Maximum water capacity of the aquifer *[300–2000 mm]* – used only if `{domain}.wt_max.txt` is not in the gridded files  
  
**Initialization**
- `"dCPI": 0.3` → Average soil moisture initialization (ignored if restart is active) *[0.1 0.9]*
- `"dWTable_init": 0.1` → Average water table initialization (ignored if restart is active) *[0.1 0.9]*
- `"iFlagRestart": 1` → Activate (1) or deactivate (0) restart from a previous model state
- `"sTimeRestart": 201804302300` → Mandatory only if restart is active - Restart data in %Y%m%d%H%M format
  
**Additional modules**
- `"iFlagWS": 0` → Activate (1) or deactivate (0) the water sources
- `"dWS": 3.678e-09` → Watertable sources average initialization *[10⁻⁶–10⁻¹² 1/t]*
- `"iFlagWDL": 0` → Activate (1) or deactivate (0) the deep fracturation
- `"dWDL": 3.678e-09` → Deep fracturation *[10⁻⁶–10⁻¹² 1/t]*

### *Paths*
The `hmc_output` folder will contain the following subfolders:

**Model Results**
- `/model_results/point/`
- `/model_results/gridded/`
- `/model_results/time_series/`

**Model States**
- `/model_state/point/`
- `/model_state/gridded/`
- `/model_state/time_series/`

In [4]:
# Load configuration from the JSON file
with open(os.path.join(workspace_path, "settings", settings_file), 'r') as f:
    config = json.load(f)

print(f"Configuration updated from settings file: {settings_file}")
path_settings = config['path']

# setup output folders
output_path={}
output_path["model_state"] = os.path.join(data_path, path_settings["hmc_output"], "model_state")
output_path["model_state_point"] = os.path.join(data_path, path_settings["hmc_output"], "model_state", "point")
output_path["model_state_gridded"] = os.path.join(data_path, path_settings["hmc_output"], "model_state", "gridded")
output_path["model_results"] = os.path.join(data_path, path_settings["hmc_output"], "model_results")
output_path["model_results_point"] = os.path.join(data_path, path_settings["hmc_output"], "model_results", "point")
output_path["model_results_gridded"] = os.path.join(data_path, path_settings["hmc_output"], "model_results", "gridded")
output_path["model_results_time_series"] = os.path.join(data_path, path_settings["hmc_output"], "model_results", "time_series")
    
for key in output_path.keys():   
    os.makedirs(output_path[key], exist_ok=True)

# Create water table max raster 
print("Write wt_max raster")
dem_path = os.path.join(input_path["data_geo_gridded"], f"{config["general"]["domain"]}.dem.txt")
wtmax_path = os.path.join(input_path["data_geo_gridded"], f"{config["general"]["domain"]}.wt_max.txt")
da = rxr.open_rasterio(dem_path)
da_const = da.where(da == -9999.0, float(config["fields"]["by_value"]["dWTableHbr"]))
da_const.rio.write_nodata(-9999.0, inplace=True)
da_const.rio.to_raster(wtmax_path, driver="AAIGrid")

print("✅ Everything is ready!")
print("Results will be saved to : " + path_settings["hmc_output"].replace(str(Path().cwd().parent),""))

Configuration updated from settings file: awash_hmc2.json
Write wt_max raster
✅ Everything is ready!
Results will be saved to : /projects/awash_iwrm/continuum/long_run2/


Everything is ready! Just set the start and end dates and wait for the model to finish its run!

In [ ]:
# Specify run dates
date_start = "2001-06-05 00:00"
date_end = "2002-08-22 23:00"

# execute Continuum model
dt_start = datetime.strptime(date_start, "%Y-%m-%d %H:%M")
dt_end   = datetime.strptime(date_end, "%Y-%m-%d %H:%M")
hours = int((dt_end - dt_start).total_seconds() / 3600)
shybox_tools.run_hmc_model(workspace_path, config, date_start, hours)

# clean the system
try:
    os.remove(str(Path().cwd().parent) + "/notebook_continuum/LogZip.txt")
except:
    pass
import shutil
if os.path.isfile(str(Path().cwd().parent) + "/notebook_continuum/hmc.log"):
    shutil.move(str(Path().cwd().parent) + "/notebook_continuum/hmc.log",
               config["path"]["hmc_data"] + "/logs/hmc.log")    
print("✅ Model run finished!")

✓ Successfully merged JSON files!
  Input:    /home/continuumuser/workdir/settings/config/app_runner_workflow_hmc_base.json
  Output:   /home/continuumuser/workdir/projects/awash_iwrm/continuum/config/app_runner_workflow_hmc_iwrn.json
Running Continuum Hydrological Model...


# 🎉 CONGRATULATION! You have run Continuum over your domain!!